# ਪਾਠ 11 - ਮਾਡਲ ਸੰਦਰਭ ਪ੍ਰੋਟੋਕੋਲ (MCP)

ਇਹ **ਮਾਡਲ ਸੰਦਰਭ ਪ੍ਰੋਟੋਕੋਲ (MCP)** ਇੱਕ ਖੁੱਲਾ ਮਿਆਰ ਹੈ ਜੋ ਏਜੰਟਾਂ ਨੂੰ ਰਨਟਾਈਮ 'ਤੇ ਗਤੀਸ਼ੀਲ ਢੰਗ ਨਾਲ ਟੂਲ, ਸਰੋਤ, ਅਤੇ ਡੇਟਾ ਸਰੋਸਾਂ ਨੂੰ ਖੋਜਣ ਅਤੇ ਵਰਤਣ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ। ਏਜੰਟ ਵਿੱਚ ਟੂਲਾਂ ਨੂੰ ਹਾਰਡਕੋਡ ਕਰਨ ਦੀ ਥਾਂ, MCP ਏਜੰਟਾਂ ਨੂੰ ਬਾਹਰੀ ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਨ ਦੀ ਆਗਿਆ ਦਿੰਦਾ ਹੈ ਜੋ ਲੋੜ ਪੈਣ 'ਤੇ ਸਮਰੱਥਾਵਾਂ ਪ੍ਰਗਟ ਕਰਦੇ ਹਨ।

ਇਸ ਪਾਠ ਵਿੱਚ, ਤੁਸੀਂ ਸਿੱਖੋਗੇ:
- MCP ਕੀ ਹੈ ਅਤੇ ਏਜੰਟ ਸਿਸਟਮਾਂ ਲਈ ਇਹ ਕਿਉਂ ਮਹੱਤਵਪੂਰਨ ਹੈ
- MCP ਕਲਾਇੰਟ-ਸਰਵਰ ਆਰਕੀਟੈਕਚਰ ਕਿਵੇਂ ਕੰਮ ਕਰਦੀ ਹੈ
- MCP-ਸਟਾਈਲ ਟੂਲ ਖੋਜ ਵਰਤਣ ਵਾਲੇ ਏਜੰਟ ਕਿਵੇਂ ਬਣਾਏ ਜਾਂਦੇ ਹਨ


## ਸੈਟਅਪ

**ਪੂਰਵ-ਸ਼ਰਤਾਂ:**
- Azure AI Foundry ਪ੍ਰੋਜੈਕਟ ਜਿਸ ਵਿੱਚ ਇੱਕ ਤੈਨਾਤ ਮਾਡਲ ਹੋਵੇ
- ਪ੍ਰਮਾਣੀਕਰਨ ਲਈ `az login` ਚਲਾਓ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) ਕੀ ਹੈ?

MCP ਬਾਹਰੀ ਟੂਲਾਂ ਅਤੇ ਡੇਟਾ ਸਰੋਤਾਂ ਨੂੰ ਖੋਜਣ ਅਤੇ ਉਨ੍ਹਾਂ ਨਾਲ ਇੰਟਰੈਕਟ ਕਰਨ ਲਈ ਇੱਕ ਮਿਆਰੀ ਤਰੀਕਾ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦਾ ਹੈ:

- **MCP Server**: ਮਿਆਰੀ ਪ੍ਰੋਟੋਕੌਲ ਰਾਹੀਂ ਟੂਲਾਂ, ਸਰੋਤਾਂ ਅਤੇ ਪ੍ਰਾਂਪਟਾਂ ਨੂੰ ਪੇਸ਼ ਕਰਦਾ ਹੈ
- **MCP Client**: ਏਜੰਟ ਰਨਟਾਈਮ ਜੋ ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਦਾ ਹੈ ਅਤੇ ਉਪਲਬਧ ਸਮਰੱਥਾਵਾਂ ਨੂੰ ਖੋਜਦਾ ਹੈ
- **Dynamic Discovery**: ਏਜੰਟਾਂ ਨੂੰ ਹਾਰਡਕੋਡ ਕੀਤੇ ਟੂਲਾਂ ਦੀ ਲੋੜ ਨਹੀਂ — ਉਹ ਰਨਟਾਈਮ 'ਤੇ ਜੋ ਉਪਲਬਧ ਹੈ ਉਹ ਖੋਜ ਲੈਂਦੇ ਹਨ

ਇਹ ਵਿਸਤਾਰਯੋਗ ਏਜੰਟ ਸਿਸਟਮ ਬਣਾਉਣ ਲਈ ਬਹੁਤ ਪ੍ਰਭਾਵਸ਼ਾਲੀ ਹੈ, ਜਿੱਥੇ ਨਵੀਆਂ ਸਮਰੱਥਾਵਾਂ ਨੂੰ ਏਜੰਟ ਕੋਡ ਬਿਨਾਂ ਸੋਧੇ ਸ਼ਾਮਿਲ ਕੀਤਾ ਜਾ ਸਕਦਾ ਹੈ।


## MCP ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. ਏਜੰਟ (MCP client) ਇੱਕ MCP ਸਰਵਰ ਨਾਲ ਜੁੜਦਾ ਹੈ
2. ਸਰਵਰ ਉਪਲੱਬਧ ਟੂਲਾਂ ਅਤੇ ਉਨ੍ਹਾਂ ਦੀਆਂ ਸਕੀਮਾਂ ਦੀ ਇੱਕ ਸੂਚੀ ਨਾਲ ਜਵਾਬ ਦਿੰਦਾ ਹੈ
3. ਫਿਰ ਏਜੰਟ ਆਪਣੇ ਤਰਕ ਦੌਰਾਨ ਕਿਸੇ ਵੀ ਪਤਾ ਲੱਗੇ ਟੂਲ ਨੂੰ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ
4. ਨਤੀਜੇ ਉਸੇ ਪ੍ਰੋਟੋਕਾਲ ਰਾਹੀਂ ਵਾਪਸ ਆਉਂਦੇ ਹਨ


## MCP ਟੂਲ ਖੋਜ ਦਾ ਅਨੁਕਰਨ

ਕਿਉਂਕਿ ਇੱਕ ਅਸਲ MCP ਸਰਵਰ ਨੂੰ ਚੱਲ ਰਹੀ ਸਰਵਰ ਪ੍ਰਕਿਰਿਆ ਦੀ ਲੋੜ ਹੁੰਦੀ ਹੈ, ਅਸੀਂ ਉਸ ਪੈਟਰਨ ਨੂੰ `@tool` ਫੰਕਸ਼ਨਾਂ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਦਰਸਾਵਾਂਗੇ ਜੋ ਉਹ ਨਕਲ ਕਰਦੀਆਂ ਹਨ ਜੋ ਇੱਕ MCP ਨਾਲ ਜੁੜੀ ਰਹਾਇਸ਼ ਸੇਵਾ ਮੁਹੱਈਆ ਕਰਵਾਏਗੀ।

ਉਤਪਾਦਨ ਵਿੱਚ, ਇਹ ਟੂਲ ਸਥਾਨਕ ਤੌਰ 'ਤੇ ਪਰਿਭਾਸ਼ਿਤ ਕਰਨ ਦੀ ਬਜਾਏ ਗਤੀਸ਼ੀਲ ਤੌਰ 'ਤੇ MCP ਸਰਵਰ ਤੋਂ ਖੋਜੇ ਜਾਣਗੇ।


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-ਸ਼ੈਲੀ ਦੇ ਟੂਲਾਂ ਨਾਲ ਇੱਕ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## ਉਤਪਾਦਨ ਵਿੱਚ MCP

ਇੱਕ ਉਤਪਾਦਨ ਵਾਤਾਵਰਣ ਵਿੱਚ, MCP ਤਾਕਤਵਰ ਪੈਟਰਨ ਨੂੰ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ:

- **ਡਾਇਨੈਮਿਕ ਸੰਦ ਖੋਜ**: ਏਜੰਟ MCP ਸਰਵਰਾਂ ਨਾਲ ਜੁੜਦੇ ਹਨ ਅਤੇ ਰਨਟਾਈਮ ਦੌਰਾਨ ਸੰਦਾਂ ਨੂੰ ਖੋਜਦੇ ਹਨ
- **ਅਲੱਗ-ਅਲੱਗ ਢਾਂਚਾ**: ਟੂਲ ਪ੍ਰਦਾਤਾ ਏਜੰਟ ਤੋਂ ਸਵਤੰਤਰ ਤੌਰ 'ਤੇ ਅਪਡੇਟ ਕਰ ਸਕਦੇ ਹਨ
- **ਸੰਗਠਨ-ਪਾਰ ਸਾਂਝਾ**: ਟੀਮਾਂ MCP ਸਰਵਰਾਂ ਰਾਹੀਂ ਸਮਰੱਥਾਵਾਂ ਪ੍ਰਗਟ ਕਰ ਸਕਦੀਆਂ ਹਨ, ਜਿਨ੍ਹਾਂ ਨੂੰ ਕੋਈ ਵੀ ਏਜੰਟ ਵਰਤ ਸਕਦਾ ਹੈ
- **Microsoft Agent Framework ਸਮਰਥਨ**: MAF ਕੋਲ `mcp` ਇੰਟਿਗਰੇਸ਼ਨ ਰਾਹੀਂ ਬਿਲਟ-ਇਨ MCP ਕਲਾਇੰਟ ਸਮਰਥਨ ਹੈ

MAF ਨਾਲ ਇੱਕ ਅਸਲ MCP ਸਰਵਰ ਵਰਤਣ ਲਈ, ਤੁਸੀਂ `hosted_mcp_tool()` ਜਾਂ MCP ਕਲਾਇੰਟ ਇੰਟਿਗਰੇਸ਼ਨ ਰਾਹੀਂ ਜੁੜੋਗੇ।

**ਹੋਰ ਜਾਣੋ:**
- [MCP ਵਿਸ਼ੇਸ਼ਣ](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP ਸਮਰਥਨ](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ, ਤੁਸੀਂ ਸਿੱਖਿਆ:
- **MCP** ਏਜੰਟਾਂ ਅਤੇ ਟੂਲ ਪ੍ਰਦਾਤਾਵਾਂ ਦੇ ਵਿਚਕਾਰ ਡਾਇਨਾਮਿਕ ਟੂਲ ਖੋਜ ਲਈ ਇੱਕ ਖੁੱਲਾ ਮਿਆਰ ਹੈ
- ਇਹ **ਕਲਾਇੰਟ-ਸਰਵਰ ਆਰਕੀਟੈਕਚਰ** ਏਜੰਟਾਂ ਨੂੰ ਚਲਣ ਸਮੇਂ ਸਮਰੱਥਾਵਾਂ ਖੋਜਣ ਦੇ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ
- MCP ਐਸੇ **ਵਿਸਤਰੀਯੋਗ, ਅਲੱਗ-ਅਲੱਗ ਏਜੰਟ ਪ੍ਰਣਾਲੀਆਂ** ਨੂੰ ਯੋਗ ਬਣਾਉਂਦਾ ਹੈ ਜਿੱਥੇ ਟੂਲਾਂ ਨੂੰ ਕੋਡ ਵਿੱਚ ਬਦਲਾਅ ਕੀਤੇ ਬਿਨਾਂ ਜੋੜਿਆ ਜਾ ਸਕਦਾ ਹੈ
- Microsoft Agent Framework ਉਤਪਾਦਨ ਲਈ **ਬਿਲਟ-ਇਨ MCP ਸਹਾਇਤਾ** ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
ਅਸਵੀਕਾਰਤਾ:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ AI ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਅਸੀਂ ਸਹੀਤਾਸ਼ੀਲਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਪਰ ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰਖੋ ਕਿ ਆਟੋਮੇਟਿਕ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਣਸਹੀਤੀਆਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਮੌਜੂਦ ਦਸਤਾਵੇਜ਼ ਨੂੰ ਪ੍ਰਮਾਣਿਕ ਸਰੋਤ ਵਜੋਂ ਹੀ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਤੋਂ ਉਤਪੰਨ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫ਼ਹਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਵਿਆਖਿਆ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
